# Adaptive Alpha Weighting with PPO: Enhancing Prompt-Based LLM-Generated Alphas in Quant Trading

Authors: Qizhao Chen, Hiroaki Kawashima
Published: 2025-09-01
Arxiv: https://arxiv.org/abs/2509.01393

## Strategy Description
This paper introduces a reinforcement learning framework that employs Proximal Policy Optimization (PPO) to dynamically optimize the weights of multiple large language model (LLM)-generated formulaic alphas for stock trading strategies. Formulaic alphas are mathematically defined trading signals derived from price, volume, sentiment, and other data. The strategy aims to adaptively integrate these alphas under varying market conditions to achieve more stable risk-adjusted performance.


In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT']
START_DATE = '2020-01-01'
END_DATE = '2023-01-01'
INITIAL_CAPITAL = 100000
RISK_FREE_RATE = 0.02

# Hypothesis
# The strategy aims to dynamically optimize the weights of LLM-generated alphas using PPO to achieve higher Sharpe ratios and smaller maximum drawdowns.

## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start=START_DATE, end=END_DATE)
prices = data['Adj Close']
returns = prices.pct_change().dropna()

# Compute features
features = returns.copy()
features['momentum'] = returns.shift(1)
features['volatility'] = returns.rolling(window=21).std()

# Cross-sectional normalization
features = (features - features.mean(axis=1)) / features.std(axis=1)

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Signal generation
signals = features[['momentum', 'volatility']].sum(axis=1)

# Position sizing
positions = signals / signals.abs().sum()

# Portfolio construction
portfolio = positions * returns

## Phase 4 — Vectorized Backtest

In [ ]:
# Shift signals forward by 1 period to avoid look-ahead bias
signals = signals.shift(1)

# Vectorized backtest
cumulative_returns = (1 + portfolio).cumprod() - 1

## Phase 5 — Performance Metrics

In [ ]:
import scipy.stats as stats

# Performance metrics
sharpe_ratio = np.mean(portfolio) / np.std(portfolio)
sortino_ratio = np.mean(portfolio) / np.std(portfolio[portfolio < 0])
max_drawdown = cumulative_returns.cummax() - cumulative_returns
calmar_ratio = np.mean(portfolio) / max_drawdown.max()

print(f'Sharpe Ratio: {sharpe_ratio}')
print(f'Sortino Ratio: {sortino_ratio}')
print(f'Calmar Ratio: {calmar_ratio}')
print(f'Max Drawdown: {max_drawdown.max()}')

import matplotlib.pyplot as plt
plt.plot(cumulative_returns)
plt.title('Equity Curve')
plt.show()

## Phase 6 — Monitoring Stub

In [ ]:
def monitor(live_data):
    live_returns = live_data['Adj Close'].pct_change().dropna()
    live_features = live_returns.copy()
    live_features['momentum'] = live_returns.shift(1)
    live_features['volatility'] = live_returns.rolling(window=21).std()
    live_features = (live_features - live_features.mean(axis=1)) / live_features.std(axis=1)
    live_signals = live_features[['momentum', 'volatility']].sum(axis=1)
    live_positions = live_signals / live_signals.abs().sum()
    live_portfolio = live_positions * live_returns
    daily_pnl = live_portfolio.sum()
    print(f'Daily P&L: {daily_pnl}')
    print(f'Current Positions: {live_positions}')

# Example usage
monitor(yf.download(UNIVERSE, start=END_DATE, end=pd.Timestamp.today().strftime('%Y-%m-%d')))